# Experimento MAREA — Competidor: **GWO**

**GWO — Grey Wolf Optimizer (Mirjalili et al., 2014)
Sin parámetros libres (a decrece de 2 a 0)**

### Protocolo experimental
- **Benchmarks:** 71 funciones CEC 2014 (30) + CEC 2017 (29) + CEC 2022 (12)
- **Dimensiones:** 10D y 50D (CEC2022: 10D y 20D)
- **MaxFES:** 5,000 / 50,000 / 500,000 / 5,000,000
- **Corridas:** 51 independientes por configuración
- **Guardado:** incremental en JSON — se puede interrumpir y reanudar

> Ejecutar las celdas en orden. Cada celda de experimento es independiente
> y guarda sus resultados automáticamente al terminar.


In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Asegurar que el directorio del proyecto esté en el path
PROJECT_DIR = os.path.dirname(os.path.abspath('__file__'))
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

from experimento_config import (
    run_experiment, load_partial, CEC_REGISTRY,
    MAXFES, DIMS, N_RUNS, POP_SIZE, RESULTS_DIR
)

ALG_NAME = 'GWO'

# Cargar el algoritmo
from competidores import gwo
def alg_fn(f, bounds, N, T, seed):
    return gwo(f, bounds, N=N, T=T, seed=seed)

print(f'✓ {ALG_NAME} cargado correctamente')
print(f'  Resultados en: {RESULTS_DIR}/')
print(f'  Funciones CEC disponibles: {len(CEC_REGISTRY)}')


## Verificación rápida (smoke test)
Corre 1 función × 3 semillas para confirmar que todo funciona antes del experimento completo.

In [ ]:
from experimento_config import get_problems_for_dim
import time

problems_10d = get_problems_for_dim(10)
test_prob    = problems_10d[0]   # F1 de CEC2014
print(f'Probando con: {test_prob["name"]} | dim={test_prob["dim"]}')

t0 = time.time()
for seed in range(3):
    r = alg_fn(test_prob['f'], test_prob['bounds'], N=POP_SIZE, T=62, seed=seed)
    print(f'  seed={seed}  best_f={r["best_f"]:.4e}')
print(f'Tiempo: {time.time()-t0:.2f}s  ✓ Listo para experimento completo')


## Experimento — MaxFES = 5,000
*Caso de uso: optimización costosa / tiempo-real*


In [ ]:
# ── MaxFES = 5,000 ──────────────────────────────────────────────
# Se corre sobre 10D y 50D. Cada corrida guarda su resultado.
# Si se interrumpe, la siguiente ejecución retoma desde donde quedó.

for ndim in DIMS:
    run_experiment(ALG_NAME, alg_fn, ndim=ndim, maxfes=5000)


## Experimento — MaxFES = 50,000
*Caso de uso: aplicaciones prácticas estándar*


In [ ]:
# ── MaxFES = 50,000 ──────────────────────────────────────────────
# Se corre sobre 10D y 50D. Cada corrida guarda su resultado.
# Si se interrumpe, la siguiente ejecución retoma desde donde quedó.

for ndim in DIMS:
    run_experiment(ALG_NAME, alg_fn, ndim=ndim, maxfes=50000)


## Experimento — MaxFES = 500,000
*Caso de uso: convención CEC estándar*


In [ ]:
# ── MaxFES = 500,000 ──────────────────────────────────────────────
# Se corre sobre 10D y 50D. Cada corrida guarda su resultado.
# Si se interrumpe, la siguiente ejecución retoma desde donde quedó.

for ndim in DIMS:
    run_experiment(ALG_NAME, alg_fn, ndim=ndim, maxfes=500000)


## Experimento — MaxFES = 5,000,000
*Caso de uso: búsqueda exploratoria profunda*


In [ ]:
# ── MaxFES = 5,000,000 ──────────────────────────────────────────────
# Se corre sobre 10D y 50D. Cada corrida guarda su resultado.
# Si se interrumpe, la siguiente ejecución retoma desde donde quedó.

for ndim in DIMS:
    run_experiment(ALG_NAME, alg_fn, ndim=ndim, maxfes=5000000)


## Resumen de resultados
Muestra el ranking promedio de este algoritmo en cada configuración.

In [ ]:
import json

print(f'\n=== RESUMEN {ALG_NAME} ===')
print(f'{"Config":<30} {"Completadas":>12} {"Mean":>14} {"Best":>14}')
print('-'*72)

for mf in MAXFES:
    for nd in DIMS:
        data = load_partial(ALG_NAME, nd, mf)
        vals = list(data.values())
        if vals:
            label = f'D={nd:2d} MaxFES={mf:>10,}'
            print(f'{label:<30} {len(vals):>12} {np.mean(vals):>14.4e} {np.min(vals):>14.4e}')
        else:
            print(f'D={nd:2d} MaxFES={mf:>10,}  (sin resultados aún)')
